In [1]:
import pandas as pd
import numpy as np


In [ ]:
home_df = pd.read_csv("../home_team.csv")   # MatchHomeTeamInfo
away_df = pd.read_csv("../away_team.csv")   # MatchAwayTeamInfo

# Keep only the columns we need
cols = ["player_id", "plays", "gender"]
home_df = home_df[cols].copy()
away_df = away_df[cols].copy()



In [6]:
all_players = pd.concat([home_df, away_df], ignore_index=True)
all_players = all_players.dropna(subset=["player_id", "plays"])


In [7]:
# ─────────────────────────────────────────────
# 4. NORMALISE player_id TYPE
# ─────────────────────────────────────────────
all_players["player_id"] = pd.to_numeric(
    all_players["player_id"], errors="coerce"
).astype("Int64")
all_players = all_players.dropna(subset=["player_id"])


In [8]:
# ─────────────────────────────────────────────
# 5. NORMALISE `plays` STRINGS
# ─────────────────────────────────────────────
all_players["plays"] = (
    all_players["plays"]
    .str.strip()
    .str.lower()
    .str.replace(r"[-_]", " ", regex=True)   # "right-handed" → "right handed"
    .str.replace(r"\s+", " ", regex=True)    # collapse multiple spaces
)

In [9]:
# Map known variants to canonical labels
handedness_map = {
    "right handed" : "Right-Handed",
    "right"        : "Right-Handed",
    "r"            : "Right-Handed",
    "left handed"  : "Left-Handed",
    "left"         : "Left-Handed",
    "l"            : "Left-Handed",
    "ambidextrous" : "Ambidextrous",
    "both"         : "Ambidextrous",
}
all_players["plays"] = all_players["plays"].map(handedness_map)

# Anything not in the map becomes NaN — treat as unknown and drop
unmapped_count = all_players["plays"].isna().sum()
if unmapped_count > 0:
    print(f"[Info] {unmapped_count:,} rows dropped — unrecognised 'plays' value")
all_players = all_players.dropna(subset=["plays"])


In [10]:
# ─────────────────────────────────────────────
# 6. NORMALISE `gender` STRINGS
# ─────────────────────────────────────────────
all_players["gender"] = (
    all_players["gender"]
    .str.strip()
    .str.upper()
)
# Keep only recognised gender values; set others to "Unknown"
all_players["gender"] = all_players["gender"].where(
    all_players["gender"].isin(["M", "F"]), other="Unknown"
)


In [11]:
# ─────────────────────────────────────────────
# 7. RESOLVE CONFLICTING HANDEDNESS PER PLAYER
# ─────────────────────────────────────────────
# A player may appear with different `plays` values across rows (data entry errors).
# Strategy: take the MODE per player_id; if tied, fall back to "Right-Handed"
# (statistically most likely and safest default).

def resolve_mode(series):
    mode = series.mode()
    return mode.iloc[0] if not mode.empty else "Right-Handed"

# Also take the mode for gender per player (should be consistent, but just in case)
player_df = (
    all_players
    .groupby("player_id")
    .agg(
        plays  =("plays",  resolve_mode),
        gender =("gender", resolve_mode),
    )
    .reset_index()
)

In [12]:
# ─────────────────────────────────────────────
# 8. FINAL DEDUPLICATION SAFETY CHECK
# ─────────────────────────────────────────────
# groupby already gives one row per player_id, but be explicit
player_df = player_df.drop_duplicates(subset=["player_id"])

print(f"Unique players after cleaning: {len(player_df):,}\n")


Unique players after cleaning: 1,146



In [13]:
# ─────────────────────────────────────────────
# 9. OVERALL HANDEDNESS DISTRIBUTION
# ─────────────────────────────────────────────
overall = (
    player_df["plays"]
    .value_counts()
    .reset_index()
)
overall.columns = ["Handedness", "Count"]
overall["Percentage (%)"] = (
    (overall["Count"] / overall["Count"].sum() * 100)
    .round(2)
)

print("=" * 45)
print("  Overall Handedness Distribution")
print("=" * 45)
print(overall.to_string(index=False))
print()


  Overall Handedness Distribution
  Handedness  Count  Percentage (%)
Right-Handed   1013           88.39
 Left-Handed    133           11.61



In [14]:
# ─────────────────────────────────────────────
# 10. BREAKDOWN BY GENDER
# ─────────────────────────────────────────────
gender_breakdown = (
    player_df[player_df["gender"].isin(["M", "F"])]
    .groupby(["gender", "plays"])
    .size()
    .reset_index(name="Count")
)

# Add percentage within each gender group
gender_breakdown["Percentage (%)"] = (
    gender_breakdown
    .groupby("gender")["Count"]
    .transform(lambda x: (x / x.sum() * 100).round(2))
)

gender_breakdown["gender"] = gender_breakdown["gender"].map({"M": "Men", "F": "Women"})
gender_breakdown.columns = ["Gender", "Handedness", "Count", "Percentage (%)"]

print("=" * 55)
print("  Handedness Distribution by Gender")
print("=" * 55)
print(gender_breakdown.to_string(index=False))

  Handedness Distribution by Gender
Gender   Handedness  Count  Percentage (%)
 Women  Left-Handed     41           10.05
 Women Right-Handed    367           89.95
   Men  Left-Handed     92           12.47
   Men Right-Handed    646           87.53
